# dispatch-back-fn-from-recipe — ex1: dispatch back fn from (recipe.func, argnum) for every parent

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dispatch-back-fn-from-recipe`. Running the final beacon cell reports progress against the `Backprop: dispatch back fn from recipe` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: dispatch back fn from recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dispatch-back-fn-from-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dispatch-back-fn-from-recipe"
DD_SUBTOPIC = "Backprop: dispatch back fn from recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Dispatch back_fn from recipe — quick refresher

Given a node with a `Recipe`, the reverse pass dispatches the right back_fn from the (forward_fn, argnum) registry:

```python
for argnum, parent in node.recipe.parents.items():
    back_fn = BACK_FUNCS.get_back_func(node.recipe.func, argnum)
    # back_fn is a function with signature (grad_out, out, *args, **kwargs)
```

Two things to internalize:
- **`recipe.parents` is the LOOP.** Iterating its `.items()` gives   `(argnum, parent_tensor)` pairs for every Tensor input — exactly the   parents whose gradient we need to compute.
- **The lookup uses `recipe.func` AND the argnum.** Symmetric and   asymmetric ops alike — `add` registers `add_back0` AND `add_back1`   (identical bodies); `div` registers `div_back0` AND `div_back1`   (different bodies). The dispatcher never asks if the op is   symmetric; it just looks up `(fwd, argnum)`.

### Exercise 1 — dispatch back fn from (recipe.func, argnum) for every parent

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the (recipe.func, argnum)-keyed dispatch pattern: iterate recipe.parents, look up the matching back_fn, return the (argnum, parent, back_fn) triples ready for the back_fn call site.
> Keywords: dispatch, recipe, argnum, back-funcs, parents-loop
> ```

**KCs targeted:** `dispatch-back-fn-from-recipe`, `parents-dict-by-argidx`

Implement `dispatch_back_fns(node, back_funcs)` — given a non-leaf MiniTensor and a `{(forward_fn, argnum): back_fn}` registry, return a list of `(argnum, parent, back_fn)` triples — one per parent in `node.recipe.parents` — that the caller can use as the input to the actual back_fn call.

**Signature.**
```python
def dispatch_back_fns(node, back_funcs) -> list[tuple]:
    ...
    # returns [(argnum_0, parent_0, back_fn_0), ...]
```

**Algorithm.**
```python
results = []
for argnum, parent in node.recipe.parents.items():
    back_fn = back_funcs[(node.recipe.func, argnum)]
    results.append((argnum, parent, back_fn))
return results
```

**Why this is its own atom.** The actual call site in `backprop` is two operations smashed together: (a) figure out WHICH back_fn to call (this drill), and (b) call it with the right args (the sibling atom). Separating the dispatch from the call clarifies the responsibility split — and lets you test the dispatch in isolation.

**Error case.** If `(node.recipe.func, argnum)` isn't in `back_funcs`, let the natural `KeyError` propagate — the caller (the main reverse-pass driver) has the context to decorate it.

**Leaf case.** Assume `node.recipe is not None` (the caller filters leaves before reaching this function).

In [ ]:
def dispatch_back_fns(node, back_funcs) -> list:
    """Return [(argnum, parent, back_fn), ...] for every parent of node."""
    raise NotImplementedError()


def _test_ex1():
    # --- back fns + registry ---
    def log_back(grad_out, out, x):
        return grad_out / x
    def mul_back0(grad_out, out, x, y):
        return grad_out * y
    def mul_back1(grad_out, out, x, y):
        return grad_out * x

    BF = {
        (t.log, 0): log_back,
        (t.multiply, 0): mul_back0,
        (t.multiply, 1): mul_back1,
    }

    # === single-parent case: c = log(b) ===
    b = MiniTensor(t.tensor([2.0]), requires_grad=True)
    c = MiniTensor(t.log(b.array), requires_grad=True)
    c.recipe = Recipe(func=t.log, args=(b.array,), kwargs={}, parents={0: b})
    triples = dispatch_back_fns(c, BF)
    assert len(triples) == 1, f'one parent → one triple, got {triples}'
    argnum, parent, back_fn = triples[0]
    assert argnum == 0
    assert parent is b
    assert back_fn is log_back

    # === two-parent case: out = x * y ===
    x = MiniTensor(t.tensor([2.0]), requires_grad=True)
    y = MiniTensor(t.tensor([3.0]), requires_grad=True)
    out = MiniTensor(x.array * y.array, requires_grad=True)
    out.recipe = Recipe(
        func=t.multiply, args=(x.array, y.array), kwargs={}, parents={0: x, 1: y}
    )
    triples = dispatch_back_fns(out, BF)
    assert len(triples) == 2, f'two parents → two triples, got {triples}'
    # Order may match parents.items() iteration order (insertion-preserving in py3.7+).
    by_argnum = {a: (p, f) for a, p, f in triples}
    assert by_argnum[0] == (x, mul_back0), f'argnum 0 dispatch wrong: {by_argnum[0]}'
    assert by_argnum[1] == (y, mul_back1), f'argnum 1 dispatch wrong: {by_argnum[1]}'

    # === diamond — same parent at BOTH argnums (z * z) ===
    z = MiniTensor(t.tensor([4.0]), requires_grad=True)
    out = MiniTensor(z.array * z.array, requires_grad=True)
    out.recipe = Recipe(
        func=t.multiply, args=(z.array, z.array), kwargs={}, parents={0: z, 1: z}
    )
    triples = dispatch_back_fns(out, BF)
    assert len(triples) == 2, 'diamond — z appears at both argnums → 2 triples'
    # Both should reference z; back_fns should differ (mul_back0 vs mul_back1).
    by_argnum = {a: (p, f) for a, p, f in triples}
    assert by_argnum[0][0] is z and by_argnum[1][0] is z, 'both parents are z'
    assert by_argnum[0][1] is mul_back0
    assert by_argnum[1][1] is mul_back1
    assert by_argnum[0][1] is not by_argnum[1][1], (
        'same op, different argnums must dispatch to different back_fns'
    )

    # === non-contiguous argnums: parents={1: a} (e.g. scalar at arg-0) ===
    # When forward was multiply(3.0, a) — first arg is a float, no parent at 0.
    a = MiniTensor(t.tensor([7.0]), requires_grad=True)
    out = MiniTensor(3.0 * a.array, requires_grad=True)
    out.recipe = Recipe(
        func=t.multiply, args=(3.0, a.array), kwargs={}, parents={1: a}
    )
    triples = dispatch_back_fns(out, BF)
    assert len(triples) == 1
    argnum, parent, back_fn = triples[0]
    assert argnum == 1, 'argnum must remain 1, NOT collapse to 0'
    assert parent is a
    assert back_fn is mul_back1, 'must dispatch to mul_back1, NOT mul_back0'

    # === missing registration → KeyError ===
    missing = MiniTensor(t.tensor([1.0]), requires_grad=True)
    missing.recipe = Recipe(
        func=t.sin, args=(t.tensor([1.0]),), kwargs={}, parents={0: missing}
    )
    raised = False
    try:
        dispatch_back_fns(missing, BF)
    except KeyError:
        raised = True
    assert raised, 'unregistered (fn, argnum) must propagate a KeyError'

    # === structural shape of the result ===
    # Each triple is exactly (argnum:int, parent:MiniTensor, back_fn:callable).
    argnum, parent, back_fn = dispatch_back_fns(c, BF)[0]
    assert isinstance(argnum, int)
    assert isinstance(parent, MiniTensor)
    assert callable(back_fn)
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def dispatch_back_fns(node, back_funcs) -> list:
    results = []
    for argnum, parent in node.recipe.parents.items():
        # Lookup uses the 2-key tuple (forward_fn, argnum).
        # KeyError propagates naturally — caller decorates.
        back_fn = back_funcs[(node.recipe.func, argnum)]
        results.append((argnum, parent, back_fn))
    return results
```

**Why a separate dispatch step from the actual back_fn call.** The reverse-pass loop in `backprop` does TWO things per parent: (a) decide WHICH back_fn applies, (b) call it with the right args. Splitting (a) into its own helper means:
- The KeyError surface lives in one place (easier to add a   diagnostic message).
- You can unit-test dispatch without running the math.
- A future change (e.g. swapping the registry for a vtable on   the op itself) only touches this function.

**Why iterate `recipe.parents`, not `recipe.args`.** `recipe.args` is ALL positional args including non-Tensors (scalars, shape tuples). `recipe.parents` is ALREADY the filtered `{argnum: Tensor}` dict — only the parents that actually need a back_fn. Iterating args would force isinstance-checks here that the wrapper already did.

**Why the argnum stays in the result.** The caller needs to know WHICH arg it's writing the grad to; for asymmetric ops (div, sub), `argnum=0` and `argnum=1` produce different tensor shapes and values. Without the argnum, the caller would have to re-derive it, defeating the dispatch.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()